## Python For Machine Learning Fall 2025
---
# Recurent Neural Network

# 1. Introduction

An RNN, or Recurrent Neural Network, is a type of artificial neural network designed to work with sequential data. Unlike traditional neural networks, RNNs have an internal memory, allowing them to "remember" information from previous inputs to influence current and future outputs.

## 1.1 Why Feed-Forward Networks Fail at Sequences

Standard feed-forward networks, like Multi-Layer Perceptrons (MLPs) or Convolutional Neural Networks (CNNs), are designed with a major assumption: **all inputs are independent of each other.**

* When you classify an image, the network doesn't care *what* image it saw before.
* When you process a row in a spreadsheet, you assume it's a standalone example.

This assumption breaks down for sequential data, such as text, speech, or time-series data (like stock prices).

**Consider the sentence:** "The movie was great, I really..."

To predict the next word ("loved", "enjoyed"), the network **must** remember the words "great" and "really". A feed-forward network processing one word at a time has no mechanism to *remember* previous words. It has no concept of order or context. This lack of **memory** makes them fundamentally unsuited for most sequence tasks.

## 1.2 Memory and a Hidden State

RNNs introduce the concept of a **hidden state**, which acts as the network's memory.

The core idea is simple: as the network processes each item in the sequence (e.g., each word), it doesn't just produce an output; it also updates its hidden state. This hidden state is then passed along to the *next* time step, carrying information from all previous steps.

**Think of it like reading a book:**
* **Input ($x_t$):** The word you are reading *right now*.
* **Hidden State ($h_{t-1}$):** Your understanding and memory of the story *up to* this word.
* **New Hidden State ($h_t$):** Your *updated* understanding, which combines your previous memory ($h_{t-1}$) with the new word ($x_t$).

This hidden state allows the network to maintain a "summary" or "context" of what it has seen so far, enabling it to make informed predictions based on past information.

## 1.3 The RNN Architecture

Architecturally, an RNN looks like a simple loop. A "chunk" of the network (often called an RNN cell) takes two inputs at each time step $t$:
1.  The input for the current time step, $x_t$.
2.  The hidden state from the previous time step, $h_{t-1}$.

It then produces two outputs:
1.  An output for the current time step, $y_t$ (this is optional, depending on the architecture).
2.  The new hidden state for the *next* time step, $h_t$.

The formula is often expressed as:
$$a_t = f(Wa_{t-1} + Ux_t + b_h)$$
$$y_t = Va_t + b_y$$

Where $f$ is an activation function (like `tanh`), and $W_{hh}$, $W_{xh}$, $W_{hy}$ are weight matrices that are **shared across all time steps**. This weight sharing is crucial; the network learns a *single* set of rules for how to update its memory, regardless of where it is in the sequence.



To visualize this for training (a process called Backpropagation Through Time), we "unroll" the loop. This reveals the RNN as a very deep feed-forward network, where each time step is a layer. The key difference is that every layer shares the exact same weights.



In [1]:
from IPython.display import Image
url='https://miro.medium.com/v2/resize:fit:4800/format:webp/1*dznTsiaHCvRc70fxWWEcgw.png'
Image(url=url)

## 1.4 The "Many-to-X" Architectures

RNNs are flexible and can be adapted to different types of sequence problems by changing how we handle inputs and outputs.

* **Many-to-One:** Takes a sequence as input and produces a single output. This is perfect for tasks where we need to understand the *entire* sequence to make a decision.
    * **Example:** Sentiment Analysis (Input: "This movie was great", Output: "Positive"). We'll build this today.

* **One-to-Many:** Takes a single input and produces a sequence as output.
    * **Example:** Image Captioning (Input: a single image vector, Output: "A cat sitting on a mat").

* **Many-to-Many (Synchronous):** Takes a sequence as input and produces a sequence of the *same length*, with outputs at each time step.
    * **Example:** Part-of-Speech Tagging (Input: "The cat sat", Output: "[Det] [Noun] [Verb]").

* **Many-to-Many (Asynchronous):** Takes a sequence as input and produces a sequence of a *different* length. This is the basis of sequence-to-sequence (Seq2Seq) models.
    * **Example:** Machine Translation (Input: "How are you?", Output: "¿Cómo estás?").

In [2]:
url='https://dotnettutorials.net/wp-content/uploads/2022/09/word-image-30700-1.png'
Image(url=url)

## 1.5 Train RNNs

Simple RNNs (like the one we'll build) are theoretically powerful but notoriously difficult to train on long sequences. The reason lies in the "unrolled" network.

When we train the network (using Backpropagation Through Time), we calculate gradients to update the weights. This gradient signal has to flow *backward* through every single time step.

Imagine the sequence has 100 words. The gradient from the 100th word has to be multiplied by the weight matrix 100 times to tell the network how to adjust its behavior for the 1st word.

* **Vanishing Gradients:** If the values in the weight matrix are small (e.g., < 1), multiplying them together many times makes the gradient signal shrink exponentially. It becomes so tiny ($0.1^{100}$) that the network effectively *cannot learn* from distant past information. This is known as the **long-term dependency problem**.

* **Exploding Gradients:** If the values are large (e.g., > 1), the gradient signal grows exponentially ($1.5^{100}$). This leads to massive, unstable weight updates, and the training process breaks down (`NaN` values appear).

This fundamental flaw of simple RNNs is the primary motivation for more advanced architectures like **Long Short-Term Memory (LSTM)** and **Gated Recurrent Units (GRU)**, which use "gates" to more effectively control the flow of information and gradients through time.

# 2. RNN Applications

## 2.1 Character-level Language Model

A **language model** is a type of artificial intelligence that learns the statistical patterns, grammar, and relationships within human language from vast amounts of text data. Its fundamental job is to calculate probability: given a sequence of words, it predicts the likelihood of the next word (or character) appearing. By learning to predict "fox" after "the quick brown," it builds a sophisticated model of how language is structured. This core ability allows language models to perform a wide range of tasks, from understanding text and translating languages to generating coherent, human-like sentences and even entire articles. In this example, we train a character-level language model to demonstrate the memory capability of an RNN.

In [18]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import string

### 2.1.1 Data Preparatin

We will use a simple, repetitive text as data so the RNN can learn quickly. You can replace this with any .txt file (for example, Shakespeare's sonnets).

In [19]:
text = """
We hold these truths to be self-evident, that all men are created equal,
that they are endowed by their Creator with certain unalienable Rights,
that among these are Life, Liberty and the pursuit of Happiness.
That to secure these rights, Governments are instituted among Men,
deriving their just powers from the consent of the governed.
"""

Build the vocabulary

In [20]:
chars = sorted(list(set(text)))
char_to_int = {ch: i for i, ch in enumerate(chars)}
int_to_char = {i: ch for i, ch in enumerate(chars)}

VOCAB_SIZE = len(chars)
print(f"Vocabulary size: {VOCAB_SIZE}")
print(f"Vocabulary: {''.join(chars)}")

Vocabulary size: 36
Vocabulary: 
 ,-.CGHLMRTWabcdefghijlmnopqrstuvwy


Convert the entire text to integers.

In [21]:
data_int = [char_to_int[ch] for ch in text]
print(f"Total characters in text: {len(data_int)}")

Total characters in text: 339


### 2.1.2 Create Dataset

Slice the text into fixed-length sequences. Here, `x` represents the input sequence, and `y` represents the target, which is `x` shifted one position to the right. For instance: `x = "hell"`, `y = "ello"`.

In [ ]:
SEQ_LENGTH = 50

sequences = []
labels = []
for i in range(len(data_int) - SEQ_LENGTH):
    seq_in = data_int[i : i + SEQ_LENGTH]
    seq_out = data_int[i + 1 : i + 1 + SEQ_LENGTH]
    sequences.append(seq_in)
    labels.append(seq_out)

Convert to tensors.

In [ ]:
X = torch.tensor(sequences, dtype=torch.long)
y = torch.tensor(labels, dtype=torch.long)

Create Dataset and DataLoader.

In [ ]:
class CharDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = CharDataset(X, y)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

### 2.1.3 Building the Model

We will implement a simple nn.RNN that includes an Embedding layer.

In [22]:
class CharRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_size):
        super(CharRNN, self).__init__()
        self.hidden_dim = hidden_dim

        # Embedding layer: one-hot to embedding
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # Simple RNN
        self.rnn = nn.RNN(embedding_dim,
                          hidden_dim,
                          num_layers=2,
                          batch_first=True,
                          dropout=0.2)

        # Hidden status to charactor
        self.fc = nn.Linear(hidden_dim, output_size)

    def forward(self, x, hidden):
        # x shape: (batch_size, seq_len)
        embedded = self.embedding(x)

        # embedded shape: (batch_size, seq_len, embedding_dim)
        # output shape: (batch_size, seq_len, hidden_dim)
        # hidden shape: (num_layers, batch_size, hidden_dim)
        output, hidden = self.rnn(embedded, hidden)

        # Each time step's output will be fed into a linear layer.
        # prediction shape: (batch_size, seq_len, vocab_size)
        prediction = self.fc(output)

        return prediction, hidden

    def init_hidden(self, batch_size, device):
        # Initialize the hidden state.
        return torch.zeros(2, batch_size, self.hidden_dim).to(device)


### 2.1.4 Training

Specific hyperpamaters.

In [23]:
EMBEDDING_DIM = 64
HIDDEN_DIM = 128
N_EPOCHS = 50

Create an instance of the model.

In [24]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CharRNN(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, VOCAB_SIZE).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

print(f"Training on device: {device}")

Training on device: cuda


In [25]:
for epoch in range(N_EPOCHS):
    epoch_loss = 0

    h = model.init_hidden(64, device) #batch_size=64

    for X_batch, y_batch in dataloader:
        if X_batch.shape[0] != 64: # skip the last batch
            continue

        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        # Detach the hidden state from the computation graph so that
        # gradients do not backpropagate to the previous batch (BPTT)
        h = h.detach()

        optimizer.zero_grad()

        # Forward process
        prediction, h = model(X_batch, h)

        # We must flatten the tensors to calculate the loss.
        # CrossEntropyLoss expects inputs of shape (N*C) and (N)
        # Prediction is (batch_size, seq_len, vocab_size)
        # y_batch is (batch_size, seq_len).
        loss = criterion(prediction.view(-1, VOCAB_SIZE), y_batch.view(-1))

        loss.backward()

        # Gradient clipping is used to prevent exploding gradients.
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        epoch_loss += loss.item()

    if (epoch + 1) % 5 == 0:
        print(f'Epoch: {epoch+1:02} | Loss: {epoch_loss/len(dataloader):.3f}')

print("Training complete.")


Epoch: 05 | Loss: 0.660
Epoch: 10 | Loss: 0.189
Epoch: 15 | Loss: 0.141
Epoch: 20 | Loss: 0.121
Epoch: 25 | Loss: 0.106
Epoch: 30 | Loss: 0.099
Epoch: 35 | Loss: 0.092
Epoch: 40 | Loss: 0.092
Epoch: 45 | Loss: 0.089
Epoch: 50 | Loss: 0.087
Training complete.


### 2.1.5 Prediction / Sampling

Let us see what the model has learned by having it generate some text.

In [26]:
def sample_text(model, start_text='That', length=100):
    print(f"--- Sampling text starting with '{start_text}' ---")
    model.eval()

    chars = [ch for ch in start_text]
    h = model.init_hidden(1, device) # batch_size=1

    # "Warm up" the hidden state by letting the model read the starting text.
    for char in chars:
        char_idx = char_to_int.get(char, 0)
        x = torch.tensor([[char_idx]], dtype=torch.long).to(device)
        output, h = model(x, h)

    # Begin generating character by character.
    # The output shape is (1, 1, vocab_size), and we take the last item.
    last_char_logits = output[0, -1, :]

    # Sampling
    p = torch.nn.functional.softmax(last_char_logits / 0.8, dim=0)
    char_idx = torch.multinomial(p, 1).item()

    # Auto-regressive generation.
    generated_text = start_text
    for _ in range(length):
        generated_text += int_to_char[char_idx]

        # The newly generated character becomes the next input.
        x = torch.tensor([[char_idx]], dtype=torch.long).to(device)
        output, h = model(x, h)

        # Next iteration
        last_char_logits = output[0, -1, :]
        p = torch.nn.functional.softmax(last_char_logits / 0.8, dim=0)
        char_idx = torch.multinomial(p, 1).item()

    print(generated_text)

# Run sampling
sample_text(model, start_text='The', length=200)
sample_text(model, start_text='Life, Liberty', length=200)

--- Sampling text starting with 'The' ---
Thecure these rights, Governments are instituted among Men,
deriving their just powers from the consent of the governed.
That to secure these rights, Governments are instituted among Men,
deriving their 
--- Sampling text starting with 'Life, Liberty' ---
Life, Liberty and the pursuit of Happiness.
That to secure these rights, Governments are instituted among Men,
deriving their just powers from the consent of the governed.
That to secure these rights, Governments 


## 2.2 Sentiment Analysis with PyTorch RNNs

We will build a **Many-to-One** RNN designed to classify short sentences as having `positive (1)` or `negative (0)` sentiment. This model will be implemented using torch, specifically leveraging `nn.Embedding`, `nn.RNN`, and `nn.Linear` from the `torch.nn` module. While industrial-strength NLP applications would typically use a dedicated library like torchtext or huggingface/datasets to manage the complex data processing pipeline (including tokenization, vocabulary building, and padding), this example will perform these steps manually on a tiny dataset. This approach keeps the focus squarely on the RNN model itself rather than the surrounding data-handling tools.

In [28]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer

### 2.2.1 Load Dataset and Tokenizer


Load the SST-2 dataset, a standard benchmark for sentiment analysis. The 'train' split contains approximately 67,000 samples, and the 'validation' split contains 872.

In [29]:
dataset = load_dataset("glue", "sst2")

Load a pre-trained tokenizer. We will use 'bert-base-uncased' because it provides an excellent vocabulary.

In [30]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

Apply the tokenizer to the entire dataset.

In [31]:
def tokenize_function(examples):
    # This will pad all sentences to the same length (padding="max_length")
    # and truncate long ones (truncation=True)
    return tokenizer(examples["sentence"], padding="max_length", truncation=True, max_length=64)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

Set the format to PyTorch tensors

In [32]:
tokenized_datasets.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

print("Dataset splits:")
print(tokenized_datasets)
print("\nExample from training data:")
print(tokenized_datasets["train"][0])

Dataset splits:
DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1821
    })
})

Example from training data:
{'label': tensor(0), 'input_ids': tensor([  101,  5342,  2047,  3595,  8496,  2013,  1996, 18643,  3197,   102,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,   

### 2.2.2 Preparing Batches

RNNs in PyTorch often expect input in a specific shape: `(sequence_length, batch_size, input_size)`.

For our data, this means:
1.  We need to convert our numericalized sentences into tensors.
2.  All sentences in a batch must be the same length. We'll pad the shorter sentences with our `<pad>` token (index 0).
3.  We'll stack them and permute the dimensions to get the shape `(seq_len, batch_size)`.

In [33]:
BATCH_SIZE = 64

train_dataloader = DataLoader(
    tokenized_datasets["train"],
    shuffle=True,
    batch_size=BATCH_SIZE
)

val_dataloader = DataLoader(
    tokenized_datasets["validation"],
    batch_size=BATCH_SIZE
)

# Let's check one batch
batch = next(iter(train_dataloader))
print("\nShape of one batch (input_ids):")
print(batch['input_ids'].shape)
print("(Batch Size, Sequence Length)")


Shape of one batch (input_ids):
torch.Size([64, 64])
(Batch Size, Sequence Length)


### 2.2.3 Building the RNN Model

Now we define our model. It will have three layers:
1.  **`nn.Embedding`:** Converts our word indices (e.g., `[5, 2, 7]`) into dense vector representations (embeddings). Each word gets its own unique vector that the network will learn.
2.  **`nn.RNN`:** The recurrent layer. It takes the sequence of embeddings and processes them one by one, updating its hidden state at each step.
3.  **`nn.Linear`:** A standard fully-connected layer. We take the **final hidden state** from the RNN (which summarizes the entire sentence) and pass it through this linear layer to get a single output (a logit) for classification.

In [34]:
class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)

        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, input_ids):
        # input_ids shape: (batch_size, seq_len)
        embedded = self.embedding(input_ids)

        # embedded shape: (batch_size, seq_len, embedding_dim)
        # outputs shape: (batch_size, seq_len, hidden_dim)
        # hidden shape: (num_layers, batch_size, hidden_dim)
        outputs, hidden = self.rnn(embedded)

        # We still take the final hidden state
        # Squeeze dim 0 to remove the 'num_layers' dimension
        prediction = self.fc(hidden.squeeze(0))
        # prediction shape: (batch_size, output_dim)

        return prediction

### 2.2.4 Train the Model

Now we define our hyperparameters, instantiate the model, and choose our loss function and optimizer.

* **Loss Function:** `nn.BCEWithLogitsLoss`. This is perfect for binary (0/1) classification. It combines a Sigmoid function with Binary Cross Entropy loss, and it's more numerically stable than doing them separately.
* **Optimizer:** `torch.optim.Adam`. A popular and effective default optimizer.

In [35]:
VOCAB_SIZE = tokenizer.vocab_size

EMBEDDING_DIM = 128
HIDDEN_DIM = 256
OUTPUT_DIM = 1

model = SentimentRNN(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM)

# Loss and Optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
criterion = criterion.to(device)
print(f"Training on device: {device}")

Training on device: cuda


Run the training loop for 10 epochs.

In [36]:
N_EPOCHS = 10

for epoch in range(N_EPOCHS):

    model.train() # Set model to training mode
    epoch_loss = 0
    epoch_acc = 0

    for batch in train_dataloader:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        labels = batch['label'].to(device)

        predictions = model(input_ids).squeeze(1)

        loss = criterion(predictions, labels.float())

        rounded_preds = torch.round(torch.sigmoid(predictions))
        correct = (rounded_preds == labels).float()
        acc = correct.sum() / len(correct)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        epoch_loss += loss.item()
        epoch_acc += acc.item()

    avg_train_loss = epoch_loss / len(train_dataloader)
    avg_train_acc = epoch_acc / len(train_dataloader)

    model.eval()
    epoch_val_loss = 0
    epoch_val_acc = 0

    with torch.no_grad():
        for batch in val_dataloader:
            input_ids = batch['input_ids'].to(device)
            labels = batch['label'].to(device)

            predictions = model(input_ids).squeeze(1)
            loss = criterion(predictions, labels.float())

            rounded_preds = torch.round(torch.sigmoid(predictions))
            correct = (rounded_preds == labels).float()
            acc = correct.sum() / len(correct)

            epoch_val_loss += loss.item()
            epoch_val_acc += acc.item()

    avg_val_loss = epoch_val_loss / len(val_dataloader)
    avg_val_acc = epoch_val_acc / len(val_dataloader)

    print(f'Epoch: {epoch+1:02}')
    print(f'\tTrain Loss: {avg_train_loss:.3f} | Train Acc: {avg_train_acc*100:.2f}%')
    print(f'\t Val. Loss: {avg_val_loss:.3f} |  Val. Acc: {avg_val_acc*100:.2f}%')

Epoch: 01
	Train Loss: 0.691 | Train Acc: 54.91%
	 Val. Loss: 0.709 |  Val. Acc: 50.89%
Epoch: 02
	Train Loss: 0.690 | Train Acc: 54.82%
	 Val. Loss: 0.697 |  Val. Acc: 50.89%
Epoch: 03
	Train Loss: 0.690 | Train Acc: 54.83%
	 Val. Loss: 0.693 |  Val. Acc: 50.89%
Epoch: 04
	Train Loss: 0.690 | Train Acc: 55.03%
	 Val. Loss: 0.700 |  Val. Acc: 50.89%
Epoch: 05
	Train Loss: 0.690 | Train Acc: 54.89%
	 Val. Loss: 0.702 |  Val. Acc: 50.89%
Epoch: 06
	Train Loss: 0.690 | Train Acc: 54.87%
	 Val. Loss: 0.695 |  Val. Acc: 49.11%
Epoch: 07
	Train Loss: 0.690 | Train Acc: 54.88%
	 Val. Loss: 0.695 |  Val. Acc: 49.11%
Epoch: 08
	Train Loss: 0.692 | Train Acc: 54.28%
	 Val. Loss: 0.704 |  Val. Acc: 50.89%
Epoch: 09
	Train Loss: 0.690 | Train Acc: 54.88%
	 Val. Loss: 0.712 |  Val. Acc: 50.89%
Epoch: 10
	Train Loss: 0.690 | Train Acc: 54.68%
	 Val. Loss: 0.699 |  Val. Acc: 50.89%


As previously mentioned, training RNNs is very challenging due to gradient problems and long-range dependency issues. We can see that on the SST-2 dataset, the simple RNN we defined does not converge, and its classification ability is similar to random guessing.